# FFAI Conditional Execution

This notebook demonstrates FFAI's conditional execution system, which allows
prompts to be **skipped** or **executed** based on the results of prior prompts.

New features covered:

1. **`condition=` parameter** -- evaluate an expression before executing a prompt
2. **`status` field** on `ResponseResult` -- "success", "skipped", or "failed"
3. **`condition_trace`** -- see the resolved condition expression for debugging
4. **`strict=` parameter** -- fail fast on typo'd `{{name.response}}` references

The condition evaluator uses a safe AST-based parser (no `eval()`) supporting
comparisons, boolean logic, string operations, and JSON path extraction.

**Note:** This notebook uses a mocked client so it runs without API keys.

In [ ]:
import sys
from pathlib import Path
from unittest.mock import MagicMock, patch

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from ffai.Clients.FFLiteLLMClient import FFLiteLLMClient  # noqa: E402
from ffai.config import Config  # noqa: E402
from ffai.core.response_options import ResponseOptions  # noqa: E402
from ffai.FFAI import FFAI  # noqa: E402

mock_config = MagicMock(spec=Config)
mock_config.paths = MagicMock()
mock_config.paths.ffai_data = "/tmp/ffai_conditional_example"

mock_client = MagicMock(spec=FFLiteLLMClient)
mock_client.model = "mistral/mistral-small-latest"
mock_client.get_conversation_history.return_value = []
mock_client.set_conversation_history = MagicMock()
mock_client.clear_conversation = MagicMock()
mock_client.last_usage = None
mock_client.last_cost_usd = 0.0

with patch("src.FFAI.get_config", return_value=mock_config):
    ffai = FFAI(mock_client)

print(f"FFAI initialized (mocked client, model={mock_client.model})")

<div class="page-break"></div>

---

## Step 1: Simulate a Prior "fetch" Step

To demonstrate conditions, we need some prior history. We'll inject a simulated
"fetch" result directly into FFAI's history, as if a previous prompt had
successfully retrieved data.

In [ ]:
ffai.history.append({
    "prompt": "Retrieve the sales data",
    "response": "Sales data retrieved: 1,234 records from Q4 2025",
    "prompt_name": "fetch",
    "timestamp": 1700000000.0,
    "model": "mistral/mistral-small-latest",
    "history": None,
    "status": "success",
})

print("Injected 'fetch' result into history")
print(f"History entries: {len(ffai.history)}")

<div class="page-break"></div>

---

## Step 2: Condition = True (Prompt Executes)

When the condition evaluates to `True`, the prompt executes normally.
The `condition` parameter accepts expressions like `{{name.property}} == "value"`.

Available properties: `status`, `response`, `attempts`, `error`, `has_response`.

In [ ]:
mock_client.generate_response.return_value = "Analysis: Q4 sales up 12% YoY"

result = ffai.generate_response(
    "Analyze the sales data",
    prompt_name="analyze",
    options=ResponseOptions(condition='{{fetch.status}} == "success"'),
)

print(f"Status:           {result.status}")
print(f"Response:         {result.response}")
print(f"Condition trace:  {result.condition_trace}")
print(f"Model called:     {mock_client.generate_response.called}")

<div class="page-break"></div>

---

## Step 3: Condition = False (Prompt is Skipped)

Now we inject a "failed" fetch result and show that the same condition
causes the prompt to be **skipped** -- the LLM is never called.

In [ ]:
ffai.history.append({
    "prompt": "Retrieve the inventory data",
    "response": "",
    "prompt_name": "fetch_inventory",
    "timestamp": 1700000001.0,
    "model": "mistral/mistral-small-latest",
    "history": None,
    "status": "failed",
})

mock_client.generate_response.reset_mock()

result = ffai.generate_response(
    "Analyze the inventory data",
    prompt_name="analyze_inventory",
    options=ResponseOptions(condition='{{fetch_inventory.status}} == "success"'),
)

print(f"Status:           {result.status}")
print(f"Response:         {result.response}")
print(f"Condition trace:  {result.condition_trace}")
print(f"Model called:     {mock_client.generate_response.called}")

<div class="page-break"></div>

---

## Step 4: Unknown Reference (Condition Fails with Error)

If the condition references a prompt name that doesn't exist, the condition
cannot be evaluated. FFAI returns `status="failed"` with an error message
instead of raising an exception.

In [ ]:
mock_client.generate_response.reset_mock()

result = ffai.generate_response(
    "Generate a report",
    prompt_name="report",
    options=ResponseOptions(condition='{{nonexistent_step.status}} == "success"'),
)

print(f"Status:           {result.status}")
print(f"Response:         {result.response}")
print(f"Condition error:  {result.condition_error}")
print(f"Model called:     {mock_client.generate_response.called}")

<div class="page-break"></div>

---

## Step 5: Compound Conditions

Conditions support boolean logic (`and`, `or`, `not`), comparison operators
(`==`, `!=`, `<`, `>`, `<=`, `>=`), and functions like `len()`, `json_get()`,
and string methods. This enables sophisticated branching logic.

In [ ]:
mock_client.generate_response.return_value = "Deep dive: The 12% increase is driven by..."
mock_client.generate_response.reset_mock()

result = ffai.generate_response(
    "Provide a deep dive on the sales analysis",
    prompt_name="deep_dive",
    options=ResponseOptions(condition='{{fetch.status}} == "success" and len({{fetch.response}}) > 10'),
)

print(f"Status:           {result.status}")
print(f"Response:         {result.response}")
print(f"Condition trace:  {result.condition_trace}")
print(f"Model called:     {mock_client.generate_response.called}")

<div class="page-break"></div>

---

## Step 6: Strict Mode -- Catching Typos in References

By default, unknown `{{name.response}}` references in prompts are silently
replaced with empty strings. With `strict=True`, a `ValueError` is raised
instead, making it easy to catch typos early.

In [ ]:
from ffai.core.prompt_utils import interpolate_prompt

prompt = "Based on {{analysys.response}}, elaborate"

result, names = interpolate_prompt(prompt, {"analysis": "some result"}, strict=False)
print(f"Non-strict:  resolved='{result}'  (typo silently removed)")

try:
    interpolate_prompt(prompt, {"analysis": "some result"}, strict=True)
except ValueError as e:
    print(f"Strict:      ValueError raised: {e}")

<div class="page-break"></div>

---

## Step 7: Review Full History with Status

Every `generate_response()` call now records a `status` field in history.
This makes it easy to audit which prompts executed, which were skipped,
and which failed.

In [ ]:

df = ffai.history_to_dataframe()

cols = [c for c in ['prompt_name', 'status', 'response', 'model'] if c in df.columns]
if not df.is_empty():
    print(f"History: {df.shape[0]} entries")
    print()
    print(df.select(cols))
else:
    print("(empty)")

<div class="page-break"></div>

---

## Summary

| Parameter | Purpose |
|-----------|---------|
| `condition=` | AST-sandboxed expression; prompt runs only if True |
| `strict=True` | Raises on unknown `{{name.response}}` references |

| `ResponseResult.status` | Meaning |
|----------------------|--------|
| `"success"` | Prompt executed and returned a response |
| `"skipped"` | Condition evaluated to False; LLM was not called |
| `"failed"` | Condition could not be evaluated (e.g., unknown name) |

| Condition operators | Examples |
|--------------------|----------|
| Comparison | `==`, `!=`, `<`, `>`, `<=`, `>=` |
| Boolean | `and`, `or`, `not` |
| Functions | `len()`, `json_get()`, `json_has()`, `int()`, `float()` |
| String | `.startswith()`, `.endswith()`, `.lower()`, `"x" in y` |